# Finaira QuasiLeak-Bench / Laya experiment

Attach the Finaira repository as a Kaggle dataset at `/kaggle/input/finaira-poc-4`, enable Internet, and select a CUDA GPU. This notebook never substitutes another model for Laya and persists raw outputs after every request.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, time
INPUT = Path('/kaggle/input/finaira-poc-4')
if not (INPUT / 'experiments/quasileak_laya').exists():
    raise RuntimeError('Attach the repository containing experiments/quasileak_laya as /kaggle/input/finaira-poc-4')
EXP = INPUT / 'experiments/quasileak_laya'
WORK = Path('/kaggle/working/finaira_quasileak_results')
WORK.mkdir(parents=True, exist_ok=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(EXP / 'requirements.txt')])
print('experiment:', EXP)
print('cuda:', __import__('torch').cuda.is_available(), __import__('torch').cuda.device_count())

In [ ]:
# Official data audit and adapter build. The repository's released manifest is used.
V = EXP / 'vendor/QuasiLeak-Bench'
DATA = V / 'data/quasileak_bench_data_full.jsonl'
if not DATA.exists():
    raise RuntimeError('The attached experiment must include the official archive extracted under vendor/QuasiLeak-Bench/data; keep the data license with it.')
subprocess.check_call([sys.executable, str(EXP/'scripts/audit_dataset.py'), '--dataset', str(DATA), '--manifest', str(V/'data/splits/split_manifest.json'), '--metadata', str(V/'data/metadata/quasileak_bench_data_full.json'), '--json-out', str(WORK/'dataset_audit.json'), '--markdown-out', str(WORK/'DATASET_AUDIT.md')])
for split, extra in [('train', ['--training-positives']), ('validation', [])]:
    subprocess.check_call([sys.executable, str(EXP/'scripts/prepare_data.py'), '--dataset', str(DATA), '--manifest', str(V/'data/splits/split_manifest.json'), '--split', split, '--out', str(WORK/f'laya_{split}.jsonl'), '--max-groups', '256', *extra])

In [ ]:
# Download the exact Laya base checkpoint; this may take several GB.
subprocess.check_call([sys.executable, str(EXP/'scripts/download_models.py'), '--model-id', 'convaiinnovations/laya', '--out-root', str(WORK/'models')])
LAYA = WORK / 'models/convaiinnovations__laya'
subprocess.call([sys.executable, str(EXP/'scripts/evaluate_laya.py'), '--dataset', str(DATA), '--manifest', str(V/'data/splits/split_manifest.json'), '--split', 'validation', '--model', str(LAYA), '--output-dir', str(WORK/'laya_base_validation'), '--device', 'cuda', '--resume'])

In [ ]:
# C1: official Laya typed-decision fine-tuning. The rolling checkpoint is resumable.
subprocess.call([sys.executable, str(EXP/'scripts/train_laya.py'), '--train', str(WORK/'laya_train.jsonl'), '--dev', str(WORK/'laya_validation.jsonl'), '--model-dir', str(LAYA), '--out', str(WORK/'laya_finetuned'), '--device', 'cuda'])
subprocess.call([sys.executable, str(EXP/'scripts/evaluate_laya.py'), '--dataset', str(DATA), '--manifest', str(V/'data/splits/split_manifest.json'), '--split', 'test', '--model', str(LAYA), '--output-dir', str(WORK/'laya_base'), '--device', 'cuda', '--resume'])
subprocess.call([sys.executable, str(EXP/'scripts/evaluate_laya.py'), '--dataset', str(DATA), '--manifest', str(V/'data/splits/split_manifest.json'), '--split', 'test', '--model', str(WORK/'laya_finetuned'), '--output-dir', str(WORK/'laya_finetuned'), '--device', 'cuda', '--resume'])

In [ ]:
# Qwen standard and augmented arms use the released QuasiLeak builder and LoRA script.
QTRAIN = WORK/'qwen_training'; QTRAIN.mkdir(exist_ok=True)
subprocess.check_call([sys.executable, str(V/'training/build_examples.py'), '--dataset', str(DATA), '--out', str(QTRAIN)])
subprocess.call([sys.executable, str(EXP/'scripts/train_qwen.py'), '--model-id', 'Qwen/Qwen3.5-4B', '--train', str(QTRAIN/'train_A.jsonl'), '--dev', str(QTRAIN/'dev_A.jsonl'), '--out', str(WORK/'qwen_standard'), '--merge'])
subprocess.call([sys.executable, str(EXP/'scripts/train_qwen.py'), '--model-id', 'Qwen/Qwen3.5-4B', '--train', str(QTRAIN/'train_B.jsonl'), '--dev', str(QTRAIN/'dev_A.jsonl'), '--out', str(WORK/'qwen_augmented'), '--epochs', '1', '--merge'])

In [ ]:
# Evaluate both locally reproduced Qwen arms and produce a table with nulls for anything not run.
for name, model, adapter in [('qwen_standard', 'Qwen/Qwen3.5-4B', str(WORK/'qwen_standard/adapter')), ('qwen_augmented', 'Qwen/Qwen3.5-4B', str(WORK/'qwen_augmented/adapter'))]:
    subprocess.call([sys.executable, str(EXP/'scripts/evaluate_qwen.py'), '--dataset', str(DATA), '--split', 'test', '--model', model, '--adapter', adapter, '--output-dir', str(WORK/name), '--resume'])
subprocess.check_call([sys.executable, str(EXP/'scripts/compare_results.py'), '--results-root', str(WORK), '--out', str(WORK/'comparison.csv')])
print((WORK/'comparison.csv').read_text())

## Resume after disconnect

Rerun the interrupted cell with `--resume`. Laya and Qwen evaluators append one flushed JSONL row and latency row per completed sample, then rebuild predictions and score only after the pass completes. Copy `/kaggle/working/finaira_quasileak_results` out before the session ends.